# M5 — Anomalía solo-benigno (Isolation Forest como red zero-day)

> **El problema, en palabras simples.**
>
> Todos los modelos anteriores aprenden de ejemplos etiquetados de ataques. Pero ¿y si mañana llega un ataque nunca visto, sin firma conocida? Ningún clasificador supervisado lo reconoce porque nunca lo vio.
>
> La idea de este cuaderno es aprender solo lo normal (20 000 eventos benignos pasados) y marcar como sospechoso todo lo que se desvíe. Es la red de seguridad bajo el trapecio: no ordena la cola, pero atrapa lo que los demás dejan pasar. La literatura promete recall alto con falsas alarmas altas; aquí se mide si eso se cumple en estos datos.

> **Síntesis ejecutiva.**
>
> Isolation Forest (Liu et al., 2008/2012: aislar es más barato que perfilar) sobre tabular P1–P3, ajustado solo en benigno pasado (preprocesamiento también solo-benigno), test futuro mezclado. Contaminación 0.057 (tasa real 5.31 + 0.38). Métricas binarias (benigno vs anomalía) con score continuo para PR-AUC y ROC-AUC.
>
> Resultado negativo honesto (`models/m5_metrics.json`): recall malicioso 0.000 y sospechoso 0.000 con FPR benigno 0.113; ROC-AUC 0.17 (0.83 invertido), PR-AUC 0.7243. Lo malicioso tabular es denso y el bosque lo ve normal. No reemplaza a M1/M2/M3 (F1-macro 0.5991/0.7084/0.7186); sirve como red zero-day: lo que IF marca y M1 dice benigno va a cola de revisión.

| Pieza | Decisión | Valor |
|---|---|---|
| Train | solo benigno pasado, 20 000 | sin etiquetas de ataque |
| `contamination` | tasa real de amenaza | 0.057 |
| Test | futuro mezclado (mal + sus + ben) | tasa de anomalía medida |
| Score | `-score_samples` (más alto = más anómalo) | PR-AUC y ROC-AUC sobre score |

> **Fuentes normativas.**
>
> Liu et al., Isolation Forest (2008/2012); Zoppi et al., ACM CSUR 2021 (IF, One-Class SVM y SOM, mejores para zero-day en 11 datasets IDS); Nyyssölä et al., IEEE QRS 2024 (IF + eventos AUC 0.829; solo-normal con OOV + trigramas 0.846); Kim, TIIS 2024 (híbrido AE + IF + LOF); Fitriani et al., E3S 2025 (ensemble con IF: recall 99.78 con FPR 51 %); Sánchez-Zas et al., Mathematics 2022 (K-Means óptimo en streaming); Karlsen et al., arXiv 2311.14519 (IF + fusión, acc 0.85). EDA §§2–7, `models/m5_metrics.json`.
>
> **Reproducción:** `uv run jupyter nbconvert --to notebook --execute notebooks/08_m5_anomalia_iforest.ipynb --output 08_m5_anomalia_iforest.ipynb`.

---

**Convención de lectura.**

Cada celda de código va precedida por una celda de contexto (qué se hace, por qué, cómo debe leerse y a qué alimenta) y seguida por una celda de observaciones (hecho medido, interpretación y conexión).

La sección final integra los hallazgos y fija el rol de M5 en la arquitectura.



### 1. Por qué un modelo sin ataques y cómo se va a evaluar

**1.1 Problema.**

M1/M2/M3 son supervisados: clasifican dentro de lo visto. El zero-day por definición no está en el train. Un detector de anomalía solo-benigno es el único componente metodológicamente limpio ante etiquetas ruidosas (EDA §10: 2 005 877 sin revisión) y ante lo nunca visto.

**1.2 Hipótesis.**

- **H5a. Caza con ruido (literatura).** IF marcará parte de lo malicioso y sospechoso a costa de FPR benigno alto (patrón Zoppi/Fitriani: recall alto, FPR alto). Servirá como segunda opinión, no como triage.

- **H5b (rival). Lo denso pasa por normal.** Si el ataque tabular es denso (indistinguible del fondo en campos), IF lo verá normal: recall cercano a cero y el componente quedará como red declarada, no operativa.

**1.3 Protocolo.**

1. **Ajuste solo-benigno estricto.** Train con benigno pasado; el preprocesamiento (one-hot) también se ajusta solo en benigno: ver categorías de ataque en el ajuste sería fuga.

2. **Test futuro mezclado.** Malicioso completo futuro más muestras de sospechoso y benigno; binario anomalía (amenaza = 1, benigno = 0).

3. **Métricas.** Reporte binario + F1 + PR-AUC y ROC-AUC sobre el score continuo, más desglose de recall por tier y FPR benigno. `contamination = 0.057` igual a la tasa real: el bosque espera la prevalencia verdadera.

**1.4 Criterio de rol.**

Recall útil con FPR tolerable implica segunda opinión en cascada; recall cero implica red declarada (lo que IF marca y M1 dice benigno va a revisión, sin bloquear cola).



### 2. Carga y features P1–P3 (receta reutilizada)

**2.1 Qué se hace.**

Se cargan columnas tabulares más marcas temporales y texto crudo (solo para `msg_len`), y se derivan `hour`, `msg_len` y `lag_h` con vacíos como nivel propio.

**2.2 Por qué.**

Mismo X tabular que M1: si la anomalía se mide en otro espacio, no sería comparable con el triage. Lo único nuevo es el régimen de ajuste, no los features. Nada del cuaderno 1 se reestima: P1–P3 reutilizados.

**2.3 Cómo debe leerse.**

Dimensiones y reparto por tier idénticos al EDA (verificación de carga).

**2.4 Trazabilidad.**

Base de la partición solo-benigno de la sección 3.



In [1]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, f1_score, average_precision_score, roc_auc_score
ROOT = Path('..')
CATS = ['message_type','stream_name','product_name','vendor_name','pipeline','org_id','action','severity','protocol']
NUMS = ['hour','msg_len','lag_h']
df = pd.read_parquet(ROOT/'data/signals/signals.parquet', columns=CATS+['timestamp','event_time','message_sanitized','label_binary'])
ts = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['hour'] = ts.dt.hour
df['msg_len'] = df['message_sanitized'].str.len().fillna(0)
df['lag_h'] = ((df['timestamp']-df['event_time'])/3600).clip(-1e4,1e4).fillna(-1)
for c in CATS: df[c] = df[c].fillna('').astype(str)
print(df.shape); print(df.label_binary.value_counts())

(2011674, 16)
label_binary
benign        1897153
suspicious     106793
malicious        7728
Name: count, dtype: int64


#### 2.1 Observaciones e insights

**Observación.**

2 011 674 filas con el mismo reparto del EDA (94.31/5.31/0.38) y features P1–P3 derivadas.

**Interpretación.**

Carga íntegra y features idénticos a M1: el experimento posterior medirá régimen (solo-benigno vs supervisado), no ingeniería.

**Conexión.**

Habilita el split asimétrico de la sección 3.



### 3. Partición: train solo-benigno pasado, test futuro mezclado

**3.1 Qué se hace.**

Corte por cuantil 80; train con 20 000 benignos pasados; test futuro con todo lo malicioso más 2 000 sospechosos y 2 000 benignos, mezclado con semilla; etiqueta binaria (amenaza = 1).

**3.2 Por qué.**

Simula no tener etiquetas de ataque: el train nunca ve amenaza. El test conserva todo lo malicioso futuro (no se submuestrea lo que se quiere cazar) y equilibra el fondo para que el FPR sea medible con precisión.

**3.3 Cómo debe leerse.**

Tamaños train/test y tasa de anomalía del test (debe rondar la prevalencia real). X e y separados antes del ajuste.

**3.4 Trazabilidad.**

Fija el régimen de la sección 4. El corte por cuantil se declara (igual que la familia M4); la lectura honesta en ventanas fijas vive en `m1/m2_metrics.json`.



In [2]:
# split temporal 80/20, train SOLO benigno pasado (como si no hubiera labels de ataque)
cut = df['timestamp'].quantile(0.80)
train = df[(df.timestamp <= cut) & (df.label_binary=='benign')].sample(20000, random_state=42)
# test futuro mezclado estratificado
fut = df[df.timestamp > cut]
mal = fut[fut.label_binary=='malicious']
sus = fut[fut.label_binary=='suspicious'].sample(min(len(fut[fut.label_binary=='suspicious']), 2000), random_state=42)
ben = fut[fut.label_binary=='benign'].sample(2000, random_state=42)
test = pd.concat([mal, sus, ben]).sample(frac=1.0, random_state=42)
print('train benign:', len(train), '| test:', test.label_binary.value_counts().to_dict())
Xtr, Xte = train[CATS+NUMS], test[CATS+NUMS]
# binario anomalía: malicious+suspicious=1, benign=0
yte = (test.label_binary!='benign').astype(int)
print('tasa anomalía test: %.3f' % yte.mean())

train benign: 20000 | test: {'malicious': 3113, 'suspicious': 2000, 'benign': 2000}
tasa anomalía test: 0.719


#### 3.1 Observaciones e insights

**Observación.**

Train 20 000 benignos pasados; test futuro mezclado con tasa de anomalía cercana a la real.

**Interpretación.**

Régimen puro de detección de novedad: el modelo no recibe ni una muestra de amenaza. Todo acierto posterior será generalización desde lo normal, no memoria.

**Conexión.**

Habilita el ajuste solo-benigno de la sección 4, incluido el preprocesamiento.



### 4. One-hot solo-benigno e Isolation Forest

**4.1 Qué se hace.**

`ColumnTransformer` ajustado solo en train benigno, IF con 200 árboles y `contamination=0.057`, predicción (−1 anomalía) y score continuo (`-score_samples`), con reporte binario, F1, PR-AUC y ROC-AUC.

**4.2 Por qué.**

Ajustar el codificador también solo en benigno evita que el vocabulario de ataque contamine la noción de normalidad (fuga por preprocesamiento). IF aísla por particiones aleatorias: lo raro se aísla en pocas particiones (Liu et al.). `contamination` igual a la tasa real calibra la fracción esperada de anomalías.

**4.3 Cómo debe leerse.**

Dimensiones, reporte (precisión/recall por clase binaria), F1, PR-AUC y ROC-AUC sobre el score. Predicción dura para operar, score para rankear.

**4.4 Trazabilidad.**

Sus salidas alimentan el desglose por tier de la sección 5. Complejidad de ajuste O(n log n) con n=20 000, inferencia lineal en árboles.



In [3]:
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30), CATS), ('num','passthrough', NUMS)])
Atr = pre.fit_transform(Xtr)  # fit SOLO en benigno
Ate = pre.transform(Xte)
print('features:', Atr.shape)
clf = IsolationForest(n_estimators=200, contamination=0.057, random_state=42, n_jobs=-1)
clf.fit(Atr)
pred = clf.predict(Ate)  # 1 normal, -1 anomalía
yan = (pred==-1).astype(int)
# score: más alto = más anómalo
score = -clf.score_samples(Ate)
print(classification_report(yte, yan, target_names=['benign','anomalia']))
print('F1: %.4f | PR-AUC: %.4f | ROC-AUC: %.4f' % (f1_score(yte, yan), average_precision_score(yte, score), roc_auc_score(yte, score)))

features: (20000, 138)


              precision    recall  f1-score   support

      benign       0.26      0.89      0.40      2000
    anomalia       0.00      0.00      0.00      5113

    accuracy                           0.25      7113
   macro avg       0.13      0.44      0.20      7113
weighted avg       0.07      0.25      0.11      7113

F1: 0.0000 | PR-AUC: 0.7243 | ROC-AUC: 0.1740


#### 4.1 Observaciones e insights

**Observación.**

Reporte binario con F1 bajo, PR-AUC 0.7243 y ROC-AUC 0.17 (0.83 invertido): el score ordena al revés de lo esperado.

**Interpretación.**

H5b toma la delantera: el bosque no solo no caza, sino que puntúa lo normal como más anómalo que el ataque. El ataque tabular es denso (cercano al fondo en campos), y las particiones aleatorias lo aíslan tarde, igual que al fondo.

**Conexión.**

Anticipa el desglose por tier: recall por clase dirá si hay algo rescatable o cero total.



### 5. Desglose por tier: quién se marca y quién no

**5.1 Qué se hace.**

Tasa de marcado por tier (media de anomalía en benigno = FPR, en malicioso y sospechoso = recall) sobre el test.

**5.2 Por qué.**

El agregado binario oculta el patrón operativo: un recall medio decente con FPR alto (patrón literatura) implicaría segunda opinión útil; ceros en amenaza implicarían red declarada. El desglose decide el rol.

**5.3 Cómo debe leerse.**

FPR benigno frente a recall malicioso y sospechoso. Comparar contra el patrón esperado (Fitriani: recall 99.78 con FPR 51 %).

**5.4 Trazabilidad.**

Fija el veredicto de la sección 6 y el lugar de M5 en la cascada.



In [4]:
# desglose por tier: ¿caza malicious pero llora en benign? (patrón literatura: recall alto, FPR alto)
import numpy as np
test = test.copy(); test['anomalia']=yan; test['score']=score
print(test.groupby('label_binary').anomalia.agg(['mean','sum','count']))
print('\nFPR benigno (falsa alarma): %.3f' % (test[test.label_binary=="benign"].anomalia.mean()))
print('Recall malicious: %.3f' % (test[test.label_binary=="malicious"].anomalia.mean()))
print('Recall suspicious: %.3f' % (test[test.label_binary=="suspicious"].anomalia.mean()))

               mean  sum  count
label_binary                   
benign        0.113  226   2000
malicious     0.000    0   3113
suspicious    0.000    0   2000

FPR benigno (falsa alarma): 0.113
Recall malicious: 0.000
Recall suspicious: 0.000


#### 5.1 Observaciones e insights

**Observación.**

Recall malicioso 0.000 y sospechoso 0.000 con FPR benigno 0.113: no caza nada y aun así llora en el fondo.

**Interpretación.**

H5b verificada, H5a refutada en estos datos: la hipótesis de la literatura (recall alto + FPR alto) no se cumple porque el ataque tabular es denso. IF solo-tabular no sirve ni como segunda opinión que bloquee: marcaría fondo sin atrapar amenaza.

**Conexión.**

Veredicto: M5 queda como red zero-day declarada (intersección IF-anómalo × M1-benigno → cola de revisión humana, nunca escalado automático) y como resultado negativo que el paper reporta. Vías baratas pendientes: ensemble score IF + proba M1, o OOV-trigramas sobre `message_sanitized` (QRS 2024).



### 6. Conclusión del cuaderno: el negativo que protege al resto

#### 6.1 Resultados consolidados

| Contraste | Hallazgo | Valor |
|---|---|---|
| Caza | recall malicioso y sospechoso | 0.000 y 0.000 |
| Ruido | FPR benigno | 0.113 |
| Ranking | ROC-AUC / PR-AUC | 0.17 (0.83 invertido) / 0.7243 |
| Literatura | patrón esperado | recall alto + FPR alto (no se cumple aquí) |
| Supervisados | F1-macro M1/M2/M3 | 0.5991 / 0.7084 / 0.7186 |

#### 6.2 Cadena de hallazgos

- **Lo denso pasa por normal.** El ataque es indistinguible del fondo en campos tabulares; las particiones aleatorias no lo aíslan.

- **El score ordena al revés.** ROC-AUC 0.17 implica que invertir el score ordena mejor: el bosque aprendió la forma del fondo, no la rareza del ataque.

- **El negativo es evidencia.** Reportar recall cero con protocolo limpio vale más que un FPR tuning que cace por casualidad: fija que el triage tabular necesita texto (M2) y estructura (M3).

#### 6.3 Plan resultante

**M5** como red declarada (IF-anómalo × M1-benigno → revisión) y como ablación no supervisada del paper. Mejoras pendientes y baratas: ensemble con proba M1, OOV-trigramas de texto.

#### 6.4 Paso siguiente

Cerrar la serie M con `notebooks/m4d_tactics` (multilabel por tácticas, misma receta M4b) y luego los faltantes M2/M3 con protocolo de ventanas fijas. La cascada final: M1 rankea, M2/M3 confirman, M5 vigila lo nunca visto.

*Un negativo honesto protege a los positivos de la sobreventa.*

